In [ ]:
# --- CÉLULA 1: Instalações (opcional) e Imports ---
# O Google Colab já traz o pandas e o plotly instalados por defeito.
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from google.colab import files

# --- CÉLULA 2: Upload do Ficheiro ---
print("Por favor, faça o upload do ficheiro CSV (ex: humanidades_digitais_def.csv):")
uploaded = files.upload()

# Obter o nome do ficheiro carregado
filename = list(uploaded.keys())[0]

# Carregar o ficheiro com o delimitador ';'
df = pd.read_csv(filename, sep=';', low_memory=False)
print(f"\nFicheiro '{filename}' carregado com sucesso! Total de linhas brutas: {len(df)}")

# --- CÉLULA 3: Processamento e Limpeza de Dados ---
# 1. Remover linhas que não tenham título nem identificador (documentos inválidos)
df = df.dropna(subset=['title', 'identifier'], how='all').copy()

# 2. Processar Ano
df['ano_limpo'] = df['year'].fillna(df['result_year']).astype(str)
df['ano_limpo'] = df['ano_limpo'].str.extract(r'(^\d{4}$)') # Garantir que só apanhamos anos com 4 dígitos

# 3. Processar Tipologia
df['tipo_limpo'] = df['document_type_normalized'].fillna(df['document_type']).fillna('Desconhecido')

# 4. Processar Instituição
df['inst_limpa'] = df['institution_normalized'].fillna(df['origin_repository']).fillna('Desconhecida')

# 5. Processar Idioma
def limpar_idioma(lang):
    if pd.isna(lang):
        return 'Não Identificado'
    lang = str(lang)[:3].upper()
    if lang in ['POR', 'PT']: return 'PT'
    if lang in ['ENG', 'EN']: return 'EN'
    if lang in ['SPA', 'ES']: return 'ES'
    if lang in ['FRA', 'FR']: return 'FR'
    if len(lang) > 3: return 'Outro'
    return lang

df['idioma_limpo'] = df['language'].apply(limpar_idioma)

# 6. Processar PDFs descarregados
df['pdf_ok'] = df['pdf_downloaded'].astype(str).str.lower() == 'true'

print("Processamento e normalização concluídos.")

# --- CÉLULA 4: Calcular e Mostrar os KPIs ---
total_docs = len(df)
total_inst = df['inst_limpa'].nunique()
total_pdfs = df['pdf_ok'].sum()
total_langs = df['idioma_limpo'].nunique()

print("\n" + "="*35)
print("📊 INDICADORES CHAVE (KPIs)")
print("="*35)
print(f"📄 Total de Documentos: {total_docs:,}".replace(',', ' '))
print(f"🏛️ Instituições Únicas: {total_inst:,}".replace(',', ' '))
print(f"📥 PDFs Obtidos:        {total_pdfs:,}".replace(',', ' '))
print(f"🌍 Idiomas Únicos:      {total_langs:,}".replace(',', ' '))
print("="*35 + "\n")

# --- CÉLULA 5: Visualizações Interativas (Gráficos) ---

# Gráfico 1: Evolução de Publicações por Ano
ano_counts = df['ano_limpo'].dropna().value_counts().sort_index().reset_index()
ano_counts.columns = ['Ano', 'Publicações']
fig1 = px.line(ano_counts, x='Ano', y='Publicações',
               title='<b>Evolução de Publicações por Ano</b>',
               markers=True,
               template="plotly_white")
fig1.update_traces(line_color='#3b82f6', marker=dict(size=8))
fig1.show()

# Gráfico 2: Tipologia de Documento (Donut)
tipo_counts = df['tipo_limpo'].value_counts()
top5_tipos = tipo_counts.head(5)
outros_tipos = pd.Series({'Outros': tipo_counts.iloc[5:].sum()}) if len(tipo_counts) > 5 else pd.Series()
tipo_final = pd.concat([top5_tipos, outros_tipos]).reset_index()
tipo_final.columns = ['Tipologia', 'Contagem']

fig2 = px.pie(tipo_final, names='Tipologia', values='Contagem', hole=0.5,
              title='<b>Tipologia de Documento</b>',
              template="plotly_white")
fig2.show()

# Gráfico 3: Top 10 Instituições de Origem
inst_counts = df['inst_limpa'].value_counts().head(10).reset_index()
inst_counts.columns = ['Instituição', 'Documentos']
# Inverter ordem para o Plotly mostrar o maior no topo num gráfico horizontal
inst_counts = inst_counts.sort_values('Documentos', ascending=True)

fig3 = px.bar(inst_counts, x='Documentos', y='Instituição', orientation='h',
              title='<b>Top 10 Instituições de Origem</b>',
              template="plotly_white")
fig3.update_traces(marker_color='#10b981')
fig3.update_layout(yaxis=dict(tickmode='linear')) # Forçar mostrar todos os rótulos
fig3.show()

# Gráfico 4: Distribuição por Idioma
lang_counts = df['idioma_limpo'].value_counts().reset_index()
lang_counts.columns = ['Idioma', 'Contagem']

fig4 = px.pie(lang_counts, names='Idioma', values='Contagem',
              title='<b>Distribuição por Idioma</b>',
              template="plotly_white")
fig4.show()

Por favor, faça o upload do ficheiro CSV (ex: humanidades_digitais_def.csv):


Saving humanidades_digitais_def.csv to humanidades_digitais_def (1).csv

Ficheiro 'humanidades_digitais_def (1).csv' carregado com sucesso! Total de linhas brutas: 41
Processamento e normalização concluídos.

📊 INDICADORES CHAVE (KPIs)
📄 Total de Documentos: 41
🏛️ Instituições Únicas: 11
📥 PDFs Obtidos:        31
🌍 Idiomas Únicos:      2



/tmp/ipykernel_1086/530752578.py:83: FutureWarning:

The behavior of array concatenation with empty entries is deprecated. In a future version, this will no longer exclude empty items when determining the result dtype. To retain the old behavior, exclude the empty entries before the concat operation.

